In [1]:
import sqlite3
import pandas as pd

df_clean = pd.read_csv("../data/processed/online_retail_cleaned.csv", parse_dates=["InvoiceDate"])

conn = sqlite3.connect("../data/processed/retail_analysis.db")
df_clean.to_sql("fact_sales", conn, if_exists="replace", index=False)

print("Table loaded into SQLite successfully")

Table loaded into SQLite successfully


In [2]:
pd.read_sql("SELECT * FROM fact_sales LIMIT 5;", conn)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [7]:
top_products_query = """
SELECT StockCode, Description, SUM(Revenue) AS total_revenue
FROM fact_sales
GROUP BY StockCode, Description
ORDER BY total_revenue DESC
LIMIT 10;
"""

pd.read_sql(top_products_query, conn)

,StockCode,Description,total_revenue
0,22423,REGENCY CAKESTAND 3 TIER,286486.30
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,252072.46
2,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60
3,M,Manual,152340.57
4,85099B,JUMBO BAG RED RETROSPOT,136980.08
5,84879,ASSORTED COLOUR BIRD ORNAMENT,127074.17
6,POST,POSTAGE,126563.04
7,47566,PARTY BUNTING,103880.23
8,23166,MEDIUM CERAMIC TOP STORAGE JAR,81416.73
9,22086,PAPER CHAIN KIT 50'S CHRISTMAS,79594.33


In [9]:
top_customers_query = """
SELECT CustomerID, SUM(Revenue) AS total_spent
FROM fact_sales
GROUP BY CustomerID
ORDER BY total_spent DESC
LIMIT 10;
"""

pd.read_sql(top_customers_query, conn)

,CustomerID,total_spent
0,18102,608821.65
1,14646,528602.52
2,14156,313946.37
3,14911,295972.63
4,17450,246973.09
5,13694,196482.81
6,17511,175603.55
7,16446,168472.50
8,16684,147142.77
9,12415,144458.37


In [12]:
monthly_sales_query = """
SELECT strftime('%Y-%m', InvoiceDate) AS sales_month,
       SUM(Revenue) AS monthly_revenue,
       COUNT(*) AS order_lines
FROM fact_sales
GROUP BY sales_month
ORDER BY sales_month;
"""
pd.read_sql(monthly_sales_query, conn)

,sales_month,monthly_revenue,order_lines
0,2009-12,686654.160,30754
1,2010-01,557319.062,21775
2,2010-02,506371.066,23367
3,2010-03,699608.991,32301
4,2010-04,594609.192,27235
5,2010-05,599985.790,28644
6,2010-06,639066.580,31188
7,2010-07,591636.740,27032
8,2010-08,604242.650,26392
9,2010-09,831615.001,34599


In [14]:
revenue_share_by_product = """
SELECT Description,
       SUM(Revenue) AS total_revenue,
       ROUND(100.0 * SUM(Revenue) / (SELECT SUM(Revenue) FROM fact_sales), 2) AS revenue_share_pct
FROM fact_sales
GROUP BY Description
ORDER BY total_revenue DESC
LIMIT 10;
"""

pd.read_sql(revenue_share_by_product, conn)

,Description,total_revenue,revenue_share_pct
0,REGENCY CAKESTAND 3 TIER,286486.30,1.61
1,WHITE HANGING HEART T-LIGHT HOLDER,252072.46,1.42
2,"PAPER CRAFT , LITTLE BIRDIE",168469.60,0.95
3,Manual,152340.57,0.86
4,JUMBO BAG RED RETROSPOT,136980.08,0.77
5,ASSORTED COLOUR BIRD ORNAMENT,127074.17,0.72
6,POSTAGE,126563.04,0.71
7,PARTY BUNTING,103880.23,0.59
8,MEDIUM CERAMIC TOP STORAGE JAR,81416.73,0.46
9,PAPER CHAIN KIT 50'S CHRISTMAS,79594.33,0.45


In [17]:
monthly_average_order_value = """
WITH monthly_customer AS (
    SELECT strftime('%Y-%m', InvoiceDate) AS sales_month,
           CustomerID,
           SUM(Revenue) AS customer_month_revenue
    FROM fact_sales
    GROUP BY sales_month, CustomerID
)
SELECT sales_month,
       ROUND(AVG(customer_month_revenue), 2) AS avg_customer_month_revenue
FROM monthly_customer
GROUP BY sales_month
ORDER BY sales_month;
"""
pd.read_sql(monthly_average_order_value, conn)

,sales_month,avg_customer_month_revenue
0,2009-12,719.01
1,2010-01,774.05
2,2010-02,655.92
3,2010-03,661.88
4,2010-04,631.22
5,2010-05,621.10
6,2010-06,613.90
7,2010-07,637.54
8,2010-08,663.27
9,2010-09,726.30


In [21]:
Customer_concentration = """
WITH customer_totals AS (
    SELECT CustomerID, SUM(Revenue) AS total_spent
    FROM fact_sales
    GROUP BY CustomerID
),
threshold AS (
    SELECT AVG(total_spent) AS avg_spent
    FROM customer_totals
)
SELECT 
    CASE 
        WHEN total_spent >= (SELECT avg_spent FROM threshold) THEN 'Above average spenders'
        ELSE 'Below average spenders'
    END AS customer_group,
    COUNT(*) AS customers,
    ROUND(SUM(total_spent), 2) AS revenue
FROM customer_totals
GROUP BY customer_group
ORDER BY revenue DESC;
"""

pd.read_sql(Customer_concentration, conn)

,customer_group,customers,revenue
0,Above average spenders,1151,13633584.23
1,Below average spenders,4727,4109844.95


In [22]:
avg_spend_query = """
SELECT ROUND(SUM(Revenue) / COUNT(DISTINCT CustomerID), 2) AS avg_spend_per_customer
FROM fact_sales;
"""

pd.read_sql(avg_spend_query, conn)

,avg_spend_per_customer
0,3018.62


In [23]:
pd.read_sql("PRAGMA table_info(fact_sales);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,Invoice,INTEGER,0,None,0
1,1,StockCode,TEXT,0,None,0
2,2,Description,TEXT,0,None,0
3,3,Quantity,INTEGER,0,None,0
4,4,InvoiceDate,TIMESTAMP,0,None,0
5,5,Price,REAL,0,None,0
6,6,CustomerID,INTEGER,0,None,0
7,7,Country,TEXT,0,None,0
8,8,Revenue,REAL,0,None,0


In [24]:
region_revenue = """
SELECT Country,
       SUM(Revenue) AS total_revenue,
       COUNT(DISTINCT CustomerID) AS customers
FROM fact_sales
GROUP BY Country
ORDER BY total_revenue DESC;
"""
pd.read_sql(region_revenue, conn)


,Country,total_revenue,customers
0,United Kingdom,1.472315e+07,5350
1,EIRE,6.216311e+05,5
2,Netherlands,5.542323e+05,22
3,Germany,4.312625e+05,107
4,France,3.552575e+05,95
5,Australia,1.699681e+05,15
6,Spain,1.091785e+05,41
7,Switzerland,1.003653e+05,22
8,Sweden,9.154972e+04,19
9,Denmark,6.986219e+04,12


In [25]:
avg_customer_spend_country = """
WITH customer_country_spend AS (
    SELECT Country,
           CustomerID,
           SUM(Revenue) AS customer_spent
    FROM fact_sales
    GROUP BY Country, CustomerID
)
SELECT Country,
       ROUND(AVG(customer_spent), 2) AS avg_customer_spent,
       COUNT(*) AS customers
FROM customer_country_spend
GROUP BY Country
ORDER BY avg_customer_spent DESC;
"""

df_avg_country = pd.read_sql(avg_customer_spend_country, conn)
df_avg_country

,Country,avg_customer_spent,customers
0,EIRE,124326.22,5
1,Singapore,25317.06,1
2,Netherlands,25192.38,22
3,Australia,11331.21,15
4,Lithuania,6553.74,1
5,Denmark,5821.85,12
6,Iceland,5633.32,1
7,Sweden,4818.41,19
8,Japan,4713.84,10
9,Norway,4623.82,13
